# EmerClinic Databases — Fake Data Inspector
Inspect all tables in **support.db** (SaaS support data) and **clinic.db** (clinic demo data) created by `db_creation.py`.

| Database | Tables |
|---|---|
| `support.db` | `accounts`, `users`, `invoices`, `tickets`, `interactions` |
| `clinic.db` | `providers`, `appointments` |

In [1]:
import sqlite3
import pandas as pd

support_conn = sqlite3.connect('support.db')
clinic_conn  = sqlite3.connect('clinic.db')

def query(sql, db='support', **kwargs):
    conn = support_conn if db == 'support' else clinic_conn
    return pd.read_sql_query(sql, conn, **kwargs)

print('Connected to support.db and clinic.db')

Connected to support.db and clinic.db


## Row counts per table

In [2]:
support_tables = ['accounts', 'users', 'invoices', 'tickets', 'interactions']
clinic_tables  = ['providers', 'appointments']

support_counts = {t: query(f'SELECT COUNT(*) AS count FROM {t}', db='support').iloc[0]['count'] for t in support_tables}
clinic_counts  = {t: query(f'SELECT COUNT(*) AS count FROM {t}', db='clinic').iloc[0]['count'] for t in clinic_tables}

rows = [(t, c, 'support.db') for t, c in support_counts.items()] + \
       [(t, c, 'clinic.db')  for t, c in clinic_counts.items()]
pd.DataFrame(rows, columns=['table', 'row_count', 'database'])

,table,row_count,database
0,accounts,5,support.db
1,users,12,support.db
2,invoices,20,support.db
3,tickets,8,support.db
4,interactions,0,support.db
5,providers,4,clinic.db
6,appointments,20,clinic.db


## Accounts
One row per clinic/business subscribed to EmerClinic. Includes plan, billing cycle, price, and status.

In [3]:
query('SELECT * FROM accounts')

,account_id,clinic_name,email,plan,billing_cycle,price,status,created_at
0,1,Martinez Group,grossamy@example.com,Premium,annual,199.2,active,2026-03-19T19:25:20.913539
1,2,Richards-Fischer,russellamy@example.org,Premium,monthly,249.0,trial,2026-03-19T19:25:20.913539
2,3,Berry LLC,danny78@example.com,Premium,annual,199.2,suspended,2026-03-19T19:25:20.913539
3,4,"Wolfe, Orozco and Vaughan",ncox@example.net,Premium,annual,199.2,active,2026-03-19T19:25:20.913539
4,5,Holmes-Winters,qadams@example.com,Premium,annual,199.2,suspended,2026-03-19T19:25:20.913539


## Users
Staff and admin users belonging to each account.

In [4]:
query('SELECT * FROM users')

,user_id,account_id,name,email,role,last_login
0,1,1,Amy Stanley,shawnbush@example.com,admin,2026-03-19T19:25:20.913539
1,2,1,Karina Ritter,simmonsjennifer@example.net,staff,2026-03-19T19:25:20.915047
2,3,1,Michele Aguilar,ovargas@example.org,staff,2026-03-19T19:25:20.915047
3,4,2,John Wood,kkim@example.com,admin,2026-03-19T19:25:20.915047
4,5,2,Tina Sutton,lorigonzales@example.com,staff,2026-03-19T19:25:20.915047
5,6,3,Kathy Moore,jaime86@example.net,staff,2026-03-19T19:25:20.915047
6,7,3,James Stanley,timothy39@example.org,staff,2026-03-19T19:25:20.915047
7,8,3,Christina Espinoza DVM,newtonolivia@example.com,admin,2026-03-19T19:25:20.916063
8,9,4,Emily Yoder,nicole48@example.com,staff,2026-03-19T19:25:20.916063
9,10,4,Bonnie Hall,thutchinson@example.org,admin,2026-03-19T19:25:20.916063


## Invoices
Subscription invoices issued to each account, with amount, status, and dates.

In [5]:
query('SELECT * FROM invoices')

,invoice_id,account_id,amount,status,issued_date,due_date,description
0,1,1,249.0,paid,2026-03-19,2026-04-03,EmerClinic Subscription #1
1,2,1,249.0,paid,2026-02-17,2026-03-04,EmerClinic Subscription #2
2,3,1,99.0,paid,2026-01-18,2026-02-02,EmerClinic Subscription #3
3,4,1,249.0,paid,2025-12-19,2026-01-03,EmerClinic Subscription #4
4,5,2,99.0,pending,2026-03-19,2026-04-03,EmerClinic Subscription #1
5,6,2,99.0,paid,2026-02-17,2026-03-04,EmerClinic Subscription #2
6,7,2,249.0,overdue,2026-01-18,2026-02-02,EmerClinic Subscription #3
7,8,2,99.0,paid,2025-12-19,2026-01-03,EmerClinic Subscription #4
8,9,3,249.0,pending,2026-03-19,2026-04-03,EmerClinic Subscription #1
9,10,3,99.0,paid,2026-02-17,2026-03-04,EmerClinic Subscription #2


In [6]:
# Invoice status breakdown
query("""
    SELECT status, COUNT(*) AS count, ROUND(SUM(amount), 2) AS total_amount
    FROM invoices
    GROUP BY status
    ORDER BY count DESC
""")

,status,count,total_amount
0,paid,10,1740.0
1,pending,8,1542.0
2,overdue,2,498.0


## Tickets
Support tickets filed by accounts, with priority, category, and status.

In [7]:
query('SELECT * FROM tickets')

,ticket_id,account_id,summary,priority,category,status,created_at
0,TKT-472ECC,4,Yourself color car score material.,high,billing,open,2026-03-19T19:25:20.916063
1,TKT-46B2A2,3,Leave into senior staff example under threat.,low,feature_request,closed,2026-03-19T19:25:20.916063
2,TKT-2CE40A,4,View large report blood smile design probably.,low,technical,open,2026-03-19T19:25:20.916063
3,TKT-6F2A1E,2,Nor thousand charge oil response better various.,medium,feature_request,open,2026-03-19T19:25:20.916063
4,TKT-5F1BCB,4,Event past material.,low,billing,open,2026-03-19T19:25:20.916063
5,TKT-5FA5A4,3,Story century protect election music throughout.,low,feature_request,in_progress,2026-03-19T19:25:20.916063
6,TKT-798BD4,5,Around design sign second.,low,technical,open,2026-03-19T19:25:20.916063
7,TKT-83BB03,2,From much score happy air his.,low,feature_request,open,2026-03-19T19:25:20.916063


In [8]:
# Ticket breakdown by status and category
query("""
    SELECT category, priority, status, COUNT(*) AS count
    FROM tickets
    GROUP BY category, priority, status
    ORDER BY category, count DESC
""")

,category,priority,status,count
0,billing,high,open,1
1,billing,low,open,1
2,feature_request,low,closed,1
3,feature_request,low,in_progress,1
4,feature_request,low,open,1
5,feature_request,medium,open,1
6,technical,low,open,2


In [9]:
# --- Interactions: agent thread logs (intent, success, summary) ---
query('SELECT * FROM interactions')

,id,thread_id,account_id,intent,success,summary,created_at


## Interactions
Agent interaction logs keyed by `thread_id` — tracks intent, success/failure, and a short summary per conversation.

---

## Clinic DB — Providers & Appointments
The tables below come from **clinic.db**.

In [10]:
# Providers — clinicians available in the demo clinic
query('SELECT * FROM providers', db='clinic')

,provider_id,name,specialty,available
0,1,Dr. Linda Henderson,General Doctor,1
1,2,Dr. Scott Garcia,Dentist,1
2,3,Dr. Ms. Andrea Chan,Orthodontist,1
3,4,Dr. Greg Diaz,Orthodontist,1


In [11]:
# All appointments
query('SELECT * FROM appointments', db='clinic')

,appointment_id,account_id,patient_name,provider_id,appointment_date,status,reason
0,1,3,Elizabeth Wilson,2,2026-03-14 19:25,scheduled,Consultation
1,2,4,Rachel Fox,3,2026-03-21 19:25,completed,Consultation
2,3,5,Amy Sanchez,4,2026-03-28 19:25,cancelled,Checkup
3,4,5,Monica Davis,4,2026-03-23 19:25,cancelled,Checkup
4,5,2,Carl Simpson,1,2026-03-13 19:25,completed,Consultation
5,6,3,James Logan,3,2026-03-15 19:25,completed,Consultation
6,7,4,Stephanie Soto,3,2026-03-24 19:25,completed,Cleaning
7,8,2,Joshua Nolan,4,2026-03-23 19:25,cancelled,Checkup
8,9,3,Mary Edwards,4,2026-03-14 19:25,cancelled,Consultation
9,10,4,Alexander Price,2,2026-03-18 19:25,scheduled,Consultation


In [12]:
# Appointments joined with provider details
query("""
    SELECT
        a.appointment_id,
        a.patient_name,
        pr.name        AS provider,
        pr.specialty,
        a.appointment_date,
        a.status,
        a.reason
    FROM appointments a
    JOIN providers pr ON a.provider_id = pr.provider_id
    ORDER BY a.appointment_date
""", db='clinic')

,appointment_id,patient_name,provider,specialty,appointment_date,status,reason
0,17,Bradley Weber,Dr. Greg Diaz,Orthodontist,2026-03-09 19:25,scheduled,Checkup
1,19,Penny Matthews,Dr. Linda Henderson,General Doctor,2026-03-10 19:25,scheduled,Cleaning
2,5,Carl Simpson,Dr. Linda Henderson,General Doctor,2026-03-13 19:25,completed,Consultation
3,1,Elizabeth Wilson,Dr. Scott Garcia,Dentist,2026-03-14 19:25,scheduled,Consultation
4,9,Mary Edwards,Dr. Greg Diaz,Orthodontist,2026-03-14 19:25,cancelled,Consultation
5,6,James Logan,Dr. Ms. Andrea Chan,Orthodontist,2026-03-15 19:25,completed,Consultation
6,20,Debbie Clark,Dr. Greg Diaz,Orthodontist,2026-03-17 19:25,completed,Cleaning
7,10,Alexander Price,Dr. Scott Garcia,Dentist,2026-03-18 19:25,scheduled,Consultation
8,15,Ashley Richardson,Dr. Scott Garcia,Dentist,2026-03-19 19:25,scheduled,Cleaning
9,13,Morgan Baldwin,Dr. Linda Henderson,General Doctor,2026-03-20 19:25,scheduled,Consultation


In [13]:
support_conn.close()
clinic_conn.close()
print('Done.')

Done.
